
<div style="background-color: lightgreen; color: black; padding: 20px; box-sizing: border-box; width: 99%;">

👋 <b>Olá! Eu sou o Dot, seu revisor de IA.</b>

🎉 <b>Parabéns! Você foi aprovado no seu projeto.</b>

Excelente trabalho ao concluir esta análise do banco de dados de livros! Seu SQL é consistentemente correto e bem organizado em todas as tarefas, desde as CTEs usadas para as perguntas sobre autores e leitores frequentes até a abordagem com subquery para contar reviews e avaliações por livro, e cada resultado é acompanhado por uma conclusão criteriosa e orientada para o negócio. Para projetos futuros, fique atento a pequenos refinamentos, como usar a chamada mais padrão `pd.read_sql`, reduzir blocos de consultas repetitivos com loops, e garantir que o tratamento de valores nulos seja aplicado de forma consistente em todas as colunas agregadas. Ótimo trabalho no geral, continue com esse nível de rigor!

Se você tiver alguma dificuldade ou erro, fique à vontade para entrar em contato com nossa equipe de suporte clicando no ícone de chat no canto superior direito ou enviando um e-mail para [helpme@tripleten.com](mailto:helpme@tripleten.com).

<hr style="margin-top: 20px; margin-bottom: 20px;">

<details>
<summary><b style="cursor: pointer;">💡 Clique aqui para ver como ler meus comentários (Legenda de cores)</b></summary>
<div style="margin-top: 15px;">
Veja meus comentários abaixo. <b>Peço gentilmente que não os mova, modifique nem exclua</b>.

Você encontrará meus comentários destacados em caixas verdes, amarelas ou vermelhas como estas:

<div class="alert alert-block alert-success">
Sucesso. Tudo foi feito corretamente.
</div>

<div class="alert alert-block alert-warning">
Observações. Algumas recomendações.
</div>

<div class="alert alert-block alert-danger">
A corrigir. O bloco requer algumas correções.
</div>
</div>
</details>

</div>


# Projeto: Análise do Banco de Dados para Aplicativo de Livros

## 1. Objetivos do Estudo
O presente estudo analisa o banco de dados de um serviço concorrente no mercado literário, abrangendo livros, autores, editoras, classificações e resenhas. Por meio de consultas formuladas em SQL, busca-se:
* Mensurar o volume de obras recentes (publicadas após o ano 2000).
* Mapear o engajamento geral através da contagem de resenhas e da nota média individual por obra.
* Mapear a editora com maior catálogo de livros consolidados (mais de 50 páginas).
* Encontrar o autor de maior nota média com base estatística representativa (mínimo de 50 avaliações por livro).
* Avaliar o comportamento de escrita dos usuários mais ativos da plataforma.

As conclusões obtidas servirão de suporte para formular a proposta de valor do novo produto.

<div class="alert alert-block alert-success">
Sua seção de objetivos está clara e bem estruturada. Você divide o estudo em cinco metas específicas que se relacionam diretamente com as tarefas abordadas mais adiante, oferecendo ao leitor um roteiro sólido antes mesmo de qualquer código ser executado.
</div>

In [1]:
# Importar bibliotecas necessárias
import pandas as pd
from sqlalchemy import create_engine

# Configurações de acesso ao banco de dados remoto da TripleTen
db_config = {
    'user': 'practicum_student',
    'pwd': 's65BlTKV3faNIGhmvJVzOqhs',
    'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
    'port': 6432,
    'db': 'data-analyst-final-project-db'
}

# Formatação da string de conexão do PostgreSQL
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

# Criação da engine com o parâmetro de SSL obrigatório
engine = create_engine(connection_string, connect_args={'sslmode': 'require'})

# Função padronizada para rodar as consultas SQL e retornar como DataFrame
def run_query(query):
    return pd.io.sql.read_sql(query, con=engine)

<div class="alert alert-block alert-warning">
A configuração da conexão com o banco de dados está bem organizada: armazenar as credenciais em um dicionário de configuração, construir a string de conexão e encapsular a execução de queries em uma função reutilizável são boas práticas que tornam o restante do notebook muito mais fácil de ler. Uma pequena observação: a função que você usa para executar queries funciona, mas a chamada mais padrão e idiomática no pandas moderno é `pd.read_sql` — vale a pena migrar para ela em projetos futuros.
</div>

## 2. Exploração Inicial das Tabelas
Abaixo são carregadas as 5 primeiras linhas de cada tabela da base relacional para verificar esquemas e tipos de dados.

In [2]:
# Inspeção da tabela books
query_preview_books = "SELECT * FROM books LIMIT 5;"
print("--- TABELA: books ---")
display(run_query(query_preview_books))

# Inspeção da tabela authors
query_preview_authors = "SELECT * FROM authors LIMIT 5;"
print("--- TABELA: authors ---")
display(run_query(query_preview_authors))

# Inspeção da tabela publishers
query_preview_publishers = "SELECT * FROM publishers LIMIT 5;"
print("--- TABELA: publishers ---")
display(run_query(query_preview_publishers))

# Inspeção da tabela ratings
query_preview_ratings = "SELECT * FROM ratings LIMIT 5;"
print("--- TABELA: ratings ---")
display(run_query(query_preview_ratings))

# Inspeção da tabela reviews
query_preview_reviews = "SELECT * FROM reviews LIMIT 5;"
print("--- TABELA: reviews ---")
display(run_query(query_preview_reviews))

--- TABELA: books ---


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


--- TABELA: authors ---


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


--- TABELA: publishers ---


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


--- TABELA: ratings ---


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


--- TABELA: reviews ---


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


<div class="alert alert-block alert-warning">
Bom trabalho cobrindo todas as cinco tabelas com uma prévia e rótulos claros para cada uma — isso satisfaz completamente o requisito de inspecionar as tabelas antes de consultá-las. Como ideia de otimização: já que essas cinco queries só diferem pelo nome da tabela, você poderia fazer um loop sobre uma lista de nomes de tabelas e reutilizar sua função de query dentro do loop, em vez de repetir o mesmo bloco cinco vezes. Não é obrigatório, mas é um bom hábito para evitar código copiado e colado.
</div>

## Tarefa 1: Número de livros lançados após 1 de janeiro de 2000
Esta consulta contabiliza quantos livros possuem a coluna `publication_date` superior a `2000-01-01`.

In [3]:
# Consulta SQL: Livros lançados após 1º de janeiro de 2000
query_1 = """
SELECT 
    COUNT(book_id) AS total_livros_apos_2000
FROM 
    books
WHERE 
    publication_date > '2000-01-01';
"""

resultado_1 = run_query(query_1)
resultado_1

,total_livros_apos_2000
0,819


<div class="alert alert-block alert-success">
Sua consulta para esta tarefa está correta: filtrar a data de publicação para ser estritamente posterior a 2000-01-01 captura com precisão o requisito, e o resultado (819) é exibido corretamente.
</div>

### Conclusão - Tarefa 1
O resultado demonstra o volume de publicações contemporâneas disponíveis no acervo do serviço concorrente. Esse número serve de base para entender a representatividade de livros modernos em relação ao total do catálogo.

<div class="alert alert-block alert-warning">
Boa interpretação do resultado como um proxy para o quão moderno é o catálogo. Para tornar a conclusão mais concreta, considere indicar explicitamente a porcentagem que isso representa em relação ao catálogo completo (819 de 1000 é cerca de 82%) em vez de apenas descrevê-la qualitativamente — isso faria com que o insight ficasse mais claro.
</div>

## Tarefa 2: Número de avaliações e classificação média para cada livro
Para cada título na tabela `books`, calculamos o total de resenhas de texto registradas (`reviews`) e a nota média atribuída pelos usuários (`ratings`). As agregações são isoladas em subconsultas para evitar o produto cartesiano decorrente de múltiplos registros em ambas as tabelas filhas.

In [4]:
# Consulta SQL: Contagem de resenhas e nota média por livro
query_2 = """
SELECT 
    b.book_id,
    b.title,
    COALESCE(rev.total_resenhas, 0) AS total_resenhas,
    ROUND(rat.nota_media::numeric, 2) AS nota_media
FROM 
    books b
LEFT JOIN (
    SELECT 
        book_id, 
        COUNT(review_id) AS total_resenhas
    FROM 
        reviews
    GROUP BY 
        book_id
) rev ON b.book_id = rev.book_id
LEFT JOIN (
    SELECT 
        book_id, 
        AVG(rating) AS nota_media
    FROM 
        ratings
    GROUP BY 
        book_id
) rat ON b.book_id = rat.book_id
ORDER BY 
    total_resenhas DESC, 
    nota_media DESC;
"""

resultado_2 = run_query(query_2)
resultado_2

,book_id,title,total_resenhas,nota_media
0,948,Twilight (Twilight #1),7,3.66
1,302,Harry Potter and the Prisoner of Azkaban (Harr...,6,4.41
2,299,Harry Potter and the Chamber of Secrets (Harry...,6,4.29
3,656,The Book Thief,6,4.26
4,734,The Glass Castle,6,4.21
...,...,...,...,...
995,387,Leonardo's Notebooks,0,4.00
996,191,Disney's Beauty and the Beast (A Little Golden...,0,4.00
997,221,Essential Tales and Poems,0,4.00
998,83,Anne Rice's The Vampire Lestat: A Graphic Novel,0,3.67


<div class="alert alert-block alert-warning">
A lógica da query aqui está bem feita: agregar reviews e ratings independentemente em subqueries evita o produto cartesiano que você corretamente antecipou, e os left joins mais o tratamento de null na contagem de reviews garantem que livros com zero reviews ainda apareçam. Uma coisa para observar: esse tratamento de null é aplicado apenas à coluna de contagem de reviews, não à coluna de média de rating — se um livro tivesse zero ratings, essa coluna apareceria vazia. Isso não afeta sua saída atual já que todo livro parece ter pelo menos um rating, mas adicionar a mesma salvaguarda ali tornaria a query mais robusta.
</div>

### Conclusão - Tarefa 2
A listagem permite identificar as obras com maior tração comunitária (alto volume de textos) em comparação com as notas médias. Isso permite separar títulos comerciais amplamente comentados daqueles com notas altas, porém avaliados por poucos usuários.

## Tarefa 3: Editora com maior número de livros com mais de 50 páginas
Filtramos os livros cujo número de páginas (`num_pages`) é estritamente superior a 50, agrupando as publicações por editora e identificando a líder de mercado no catálogo de obras completas.

In [5]:
# Consulta SQL: Editora líder em publicações acima de 50 páginas
query_3 = """
SELECT 
    p.publisher,
    COUNT(b.book_id) AS total_livros
FROM 
    publishers p
JOIN 
    books b ON p.publisher_id = b.publisher_id
WHERE 
    b.num_pages > 50
GROUP BY 
    p.publisher
ORDER BY 
    total_livros DESC
LIMIT 1;
"""

resultado_3 = run_query(query_3)
resultado_3

,publisher,total_livros
0,Penguin Books,42


<div class="alert alert-block alert-success">
A sua query aqui está corretamente construída — filtrando os livros com mais de 50 páginas, unindo (join) publishers a books, agrupando e ordenando para encontrar a editora com mais livros. O resultado (Penguin Books, 42 livros) está correto e responde à tarefa diretamente com um join limpo.
</div>

### Conclusão - Tarefa 3
A editora identificada no topo do ranking concentra a maior fatia de títulos de grande extensão cadastrados. Para o novo aplicativo, essa editora representa a principal candidata a parcerias de licenciamento e acordos comerciais prioritários.

## Tarefa 4: Autor com maior nota média (apenas livros com 50+ avaliações)
Filtramos primeiro as obras que possuem volume relevante de notas (`COUNT(ratings.rating_id) >= 50`) para evitar distorções de notas máximas vindas de amostras pequenas. Em seguida, calculamos a nota média por autor e selecionamos o primeiro colocado.

In [6]:
# Consulta SQL: Autor com maior classificação média em obras com 50+ notas
query_4 = """
WITH livros_qualificados AS (
    SELECT 
        b.book_id,
        b.author_id,
        AVG(r.rating) AS media_livro
    FROM 
        books b
    JOIN 
        ratings r ON b.book_id = r.book_id
    GROUP BY 
        b.book_id, 
        b.author_id
    HAVING 
        COUNT(r.rating_id) >= 50
)
SELECT 
    a.author,
    ROUND(AVG(lq.media_livro)::numeric, 2) AS nota_media_autor
FROM 
    livros_qualificados lq
JOIN 
    authors a ON lq.author_id = a.author_id
GROUP BY 
    a.author
ORDER BY 
    nota_media_autor DESC
LIMIT 1;
"""

resultado_4 = run_query(query_4)
resultado_4

,author,nota_media_autor
0,J.K. Rowling/Mary GrandPré,4.28


<div class="alert alert-block alert-success">
Excelente trabalho nesta tarefa. A CTE filtra corretamente os livros com pelo menos 50 avaliações, e unir isso aos autores e calcular a média por autor é exatamente a abordagem correta. Usar uma CTE aqui torna a lógica muito mais fácil de acompanhar do que subconsultas aninhadas, e o resultado (J.K. Rowling/Mary GrandPré, 4.28) está correto.
</div>

### Conclusão - Tarefa 4
O autor apontado na consulta não apenas possui notas elevadas, mas teve seu desempenho sustentado por uma base sólida de leitores. Suas obras servem como indicação imediata para a tela inicial do novo produto e para campanhas de marketing de aquisição.

## Tarefa 5: Média de resenhas entre usuários que classificaram mais de 50 livros
Identificamos a coorte de usuários frequentes (aqueles que atribuíram nota a mais de 50 obras na tabela `ratings`). Para esse subgrupo, contabilizamos a quantidade de resenhas textuais escritas na tabela `reviews` e extraímos a média geral do grupo.

In [7]:
# Consulta SQL: Média de resenhas escritas por usuários frequentes (> 50 avaliações)
query_5 = """
WITH usuarios_frequentes AS (
    SELECT 
        username
    FROM 
        ratings
    GROUP BY 
        username
    HAVING 
        COUNT(rating_id) > 50
),
resenhas_por_usuario AS (
    SELECT 
        uf.username,
        COUNT(rev.review_id) AS total_resenhas
    FROM 
        usuarios_frequentes uf
    LEFT JOIN 
        reviews rev ON uf.username = rev.username
    GROUP BY 
        uf.username
)
SELECT 
    ROUND(AVG(total_resenhas)::numeric, 2) AS media_resenhas_usuarios_ativos
FROM 
    resenhas_por_usuario;
"""

resultado_5 = run_query(query_5)
resultado_5

,media_resenhas_usuarios_ativos
0,24.33


<div class="alert alert-block alert-success">
Ótimo trabalho nesta tarefa. A estrutura em duas etapas separa claramente a identificação dos avaliadores frequentes da contagem de suas avaliações, e o uso de um left join (em vez de um inner join) mantém corretamente os avaliadores frequentes com zero avaliações na média, ao invés de descartá-los, o que teria enviesado o resultado para cima.
</div>

### Conclusão - Tarefa 5
A média apurada evidencia o perfil comportamental dos usuários engajados: leitores que consomem e avaliam muitos títulos também demonstram propensão a escrever opiniões e textos críticos, validando a inclusão de seções sociais de comentários no novo aplicativo.

## 3. Conclusões Finais e Recomendações Estratégicas
1. **Acervo:** O catálogo demonstra foco em literatura moderna, o que direciona o portfólio inicial do aplicativo para obras contemporâneas.
2. **Parcerias de Conteúdo:** A editora líder em livros com mais de 50 páginas deve ser o primeiro contato para acordos de licenciamento.
3. **Curadoria:** O autor de destaque com alta aprovação em amostras grandes representa o benchmark de recomendação para engajamento inicial na plataforma.
4. **Funcionalidades do App:** Os leitores frequentes geram conteúdo orgânico em texto; logo, recursos de resenhas detalhadas e perfis de leitores devem ser integrados como diferenciais do produto.

<div class="alert alert-block alert-warning">
Este resumo final faz um ótimo trabalho ao traduzir o resultado de cada consulta em uma recomendação concreta e orientada aos negócios, conectando toda a análise de volta à proposta de valor do produto. Um pequeno ponto de rigor: o primeiro ponto afirma que o catálogo mostra um foco em literatura moderna baseando-se apenas na contagem pós-2000, sem comparar explicitamente com o total pré-2000 no texto — detalhar essa comparação tornaria a afirmação mais solidamente fundamentada.
</div>